# Klasifikasi DemogPairs Menggunakan ViT (Emosi dan Wajah) & XGBoost

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

C:\Users\Rezky\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images/able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images/able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images/able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images/able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images/able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images/zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images/zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images/zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images/zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
vggface_features = joblib.load('features/demogpairs_vit-face.pkl')
emotion_features = joblib.load('features/demogpairs_vit-emotion.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(emotion_features[key]) + list(vggface_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

100%|█████████████████████████████████████████████████████████████████████████| 10800/10800 [00:00<00:00, 11433.21it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'classifier': [XGBClassifier(n_jobs=1, random_state=42)],
        'classifier__tree_method': ['approx', 'hist'],
        'classifier__max_depth': [3, 6, 8],
        'classifier__gamma': [0.0, 0.1, 0.3],
        'classifier__min_child_weight': [1, 3, 5],
    }
]

pipeline = Pipeline(steps=[
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
    'roc_auc_ovr': 'roc_auc_ovr'
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.8),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

XGBClassifier: 54 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    model_prefix='models/clf_demogpairs_xgb_vit-emotion-face_',
    results_path='results/demogpairs_xgb_vit-emotion-face_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Mengevaluasi model: XGBClassifier
Fitting 5 folds for each of 54 candidates, totalling 270 fits


{'classifier': 'XGBClassifier', 'classifier__gamma': 0.0, 'classifier__max_depth': 3, 'classifier__min_child_weight': 1, 'classifier__tree_method': 'approx'}


Accuracy  : 0.8986111111111111
Precision : 0.8989110991050634
Recall    : 0.8986111111111111
F1 Score  : 0.8985784944210459
              precision    recall  f1-score   support

           0       0.94      0.91      0.93       360
           1       0.90      0.91      0.91       360
           2       0.87      0.89      0.88       360
           3       0.92      0.94      0.93       360
           4       0.91      0.87      0.89       360
           5       0.87      0.86      0.86       360

    accuracy                           0.90      2160
   macro avg       0.90      0.90      0.90      2160
weighted avg       0.90      0.90      0.90      2160



model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
XGBClassifier,models/clf_demogpairs_xgb_vit-emotion-face_XGBClassifier.pkl,"{'classifier': 'XGBClassifier', 'classifier__gamma': 0.0, 'classifier__max_depth': 3, 'classifier__min_child_weight': 1, 'classifier__tree_method': 'approx'}",0.8986111111111111,0.8985784944210459,0.8989110991050634,0.8986111111111111,54


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_xgb_vit-emotion-face_XGBClassifier.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 30985.0,
 'days': 0,
 'hours': 8,
 'minutes': 36,
 'seconds': 25.0,
 'text': '0 hari 8 jam 36 menit 25.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 271443.0,
 'days': 3,
 'hours': 3,
 'minutes': 24,
 'seconds': 3.0,
 'text': '3 hari 3 jam 24 menit 3.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'XGBClassifier', 'classifier__gamma': 0.0, 'classifier__max_depth': 3, 'classifier__min_child_weight': 1, 'classifier__tree_method': 'approx'}",0.9103,0.9005,0.8912,0.8999,0.8976,0.8999,0.8998,0.9004,0.8999,1639.0145
2,"{'classifier': 'XGBClassifier', 'classifier__gamma': 0.0, 'classifier__max_depth': 3, 'classifier__min_child_weight': 5, 'classifier__tree_method': 'hist'}",0.9034,0.9086,0.8866,0.8976,0.8999,0.8992,0.8992,0.8996,0.8992,155.2449
3,"{'classifier': 'XGBClassifier', 'classifier__gamma': 0.0, 'classifier__max_depth': 3, 'classifier__min_child_weight': 3, 'classifier__tree_method': 'approx'}",0.9074,0.9039,0.8877,0.897,0.8981,0.8988,0.8988,0.8993,0.8988,1644.0097
4,"{'classifier': 'XGBClassifier', 'classifier__gamma': 0.1, 'classifier__max_depth': 3, 'classifier__min_child_weight': 1, 'classifier__tree_method': 'hist'}",0.9103,0.897,0.8872,0.897,0.9028,0.8988,0.8988,0.8993,0.8988,161.1778
...,...,...,...,...,...,...,...,...,...,...,...
51,"{'classifier': 'XGBClassifier', 'classifier__gamma': 0.3, 'classifier__max_depth': 8, 'classifier__min_child_weight': 3, 'classifier__tree_method': 'approx'}",0.8964,0.8912,0.8721,0.8848,0.8872,0.8863,0.8861,0.8868,0.8863,1823.7147
52,"{'classifier': 'XGBClassifier', 'classifier__gamma': 0.1, 'classifier__max_depth': 8, 'classifier__min_child_weight': 1, 'classifier__tree_method': 'approx'}",0.8912,0.8843,0.8715,0.886,0.89,0.8846,0.8843,0.8851,0.8846,1949.0622
53,"{'classifier': 'XGBClassifier', 'classifier__gamma': 0.3, 'classifier__max_depth': 8, 'classifier__min_child_weight': 1, 'classifier__tree_method': 'hist'}",0.8947,0.8843,0.8744,0.8796,0.8883,0.8843,0.884,0.8846,0.8843,245.1453
54,"{'classifier': 'XGBClassifier', 'classifier__gamma': 0.3, 'classifier__max_depth': 8, 'classifier__min_child_weight': 1, 'classifier__tree_method': 'approx'}",0.8947,0.8848,0.8652,0.8791,0.8814,0.881,0.8807,0.8814,0.881,1861.2983
